In [1]:
import pandas as pd
import matplotlib.figure as figure
import matplotlib.pyplot as plt
import numpy as np
from dcekit.validation import double_cross_validation, DCEGridSearchCV
from sklearn import tree, metrics, datasets
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_predict

In [2]:
dataset = pd.read_csv('HEA42_xrd_0010_XYZ.csv', index_col=0, header=0)

In [3]:
# データ分割
y = dataset.iloc[:, 0]
x = dataset.iloc[:, 1:]

In [4]:
estimated_y_in_outer_cv = y.copy() # 外側の CV による y の推定結果を格納する変数

In [5]:
estimated_y_in_outer_cv # 確認

1     1
2     1
3     1
4     1
5     1
6     1
10    1
11    1
13    1
14    1
18    1
19    1
20    1
21    1
22    1
23    1
24    1
25    0
27    1
28    1
29    0
30    0
31    0
32    0
33    0
34    1
35    0
36    0
37    0
38    0
39    1
40    1
41    0
42    1
43    0
45    0
46    1
47    0
48    1
49    1
50    1
51    1
Name: XRD, dtype: int64

In [6]:
x # 確認

,N,M,I,T,P,N*M,N*I,N*T,N*P,M*I,...,M*I*T,M*I*P,M*T*P,I*T*P,N/M/I,N/M/T,N/M/P,N/I/T,N/I/P,N/T/P
1,0.5,50,5.00,225.0,8.5,25.0,2.500,112.50,4.25,250.00,...,56250.000,2125.00,95625.00,9562.500,0.002000,0.000044,0.001176,0.000444,0.011765,0.000261
2,0.5,50,5.00,265.3,8.1,25.0,2.500,132.65,4.05,250.00,...,66325.000,2025.00,107446.50,10744.650,0.002000,0.000038,0.001235,0.000377,0.012346,0.000233
3,0.5,70,7.00,259.6,8.0,35.0,3.500,129.80,4.00,490.00,...,127204.000,3920.00,145376.00,14537.600,0.001020,0.000028,0.000893,0.000275,0.008929,0.000241
4,0.5,40,4.00,266.0,8.1,20.0,2.000,133.00,4.05,160.00,...,42560.000,1296.00,86184.00,8618.400,0.003125,0.000047,0.001543,0.000470,0.015432,0.000232
5,1.0,70,7.00,261.6,8.1,70.0,7.000,261.60,8.10,490.00,...,128184.000,3969.00,148327.20,14832.720,0.002041,0.000055,0.001764,0.000546,0.017637,0.000472
6,5.0,70,7.00,260.4,8.0,350.0,35.000,1302.00,40.00,490.00,...,127596.000,3920.00,145824.00,14582.400,0.010204,0.000274,0.008929,0.002743,0.089286,0.002400
10,5.0,70,9.99,258.8,8.0,350.0,49.950,1294.00,40.00,699.30,...,180978.840,5594.40,144928.00,20683.296,0.007150,0.000276,0.008929,0.001934,0.062563,0.002415
11,5.0,70,7.00,269.5,9.5,350.0,35.000,1347.50,47.50,490.00,...,132055.000,4655.00,179217.50,17921.750,0.010204,0.000265,0.007519,0.002650,0.075188,0.001953
13,5.0,100,7.00,257.6,8.0,500.0,35.000,1288.00,40.00,700.00,...,180320.000,5600.00,206080.00,14425.600,0.007143,0.000194,0.006250,0.002773,0.089286,0.002426
14,5.0,100,7.00,256.3,9.5,500.0,35.000,1281.50,47.50,700.00,...,179410.000,6650.00,243485.00,17043.950,0.007143,0.000195,0.005263,0.002787,0.075188,0.002054


In [7]:
from sklearn.model_selection import StratifiedKFold
cv_shuffle = StratifiedKFold(n_splits=5, shuffle=True, random_state=3)

In [8]:
inner_fold_number = 5  # "fold_number"-fold cross-validation (CV) for inter CV 
outer_fold_number = x.shape[0]-1  # "fold_number"-fold CV for outer CV

In [9]:
param = {'max_depth':[1, 2, 3, 4, 5], 'min_samples_leaf':[1, 2, 3], 'min_samples_split':[2, 3, 4]}

In [10]:
inner_cv = DCEGridSearchCV(tree.DecisionTreeClassifier(), param, cv=cv_shuffle)
estimated_y = double_cross_validation(gs_cv=inner_cv, x=x, y=y, outer_fold_number=outer_fold_number,
                                      do_autoscaling=False, random_state=3)

1 / 41
2 / 41
3 / 41
4 / 41
5 / 41
6 / 41
7 / 41
8 / 41
9 / 41
10 / 41
11 / 41
12 / 41
13 / 41
14 / 41
15 / 41
16 / 41
17 / 41
18 / 41
19 / 41
20 / 41
21 / 41
22 / 41
23 / 41
24 / 41
25 / 41
26 / 41
27 / 41
28 / 41
29 / 41
30 / 41
31 / 41
32 / 41
33 / 41
34 / 41
35 / 41
36 / 41
37 / 41
38 / 41
39 / 41
40 / 41
41 / 41


In [11]:
estimated_y = pd.DataFrame(estimated_y)
estimated_y.to_csv('estimated_y.csv')

In [12]:
# confusion matrix between actual y and estimated y
from sklearn import metrics
confusion_matrix_train = metrics.confusion_matrix(y, estimated_y, labels=sorted(set(y)))

In [13]:
print(sorted(set(y)))

[0, 1]


In [14]:
print(confusion_matrix_train)

[[11  3]
 [ 3 25]]
